##### **Creating Patch Foundation Model**

In [1]:
# Get huggingface path and output dimension of the selected patch foundation model.
from piano.model.patch_encoder import get_model_output_dim, get_model_hf_path

model_name = "pathorchestra"

model_hf_path = get_model_hf_path(model_name=model_name)
model_output_dim = get_model_output_dim(model_name=model_name)

print(model_hf_path, model_output_dim)

hf-hub:AI4Pathology/PathOrchestra 1024


In [2]:
# Create patch foundation model.
from piano.model.patch_encoder import create_patch_encoder

patch_pfm = create_patch_encoder(model_name=model_name)

print(patch_pfm)

PathOrchestraModel(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (act): GELU(a

##### **Creating Multiple Instance Learning (MIL) Model**



In [3]:
# Create MIL model.
from piano.model.mil_factory import create_mil_model

model_name = 'wikg'

mil_model = create_mil_model(model_name=model_name)

print(mil_model)


WiKG(
  (_fc1): Sequential(
    (0): Linear(in_features=384, out_features=192, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
  )
  (W_head): Linear(in_features=192, out_features=192, bias=True)
  (W_tail): Linear(in_features=192, out_features=192, bias=True)
  (linear1): Linear(in_features=192, out_features=192, bias=True)
  (linear2): Linear(in_features=192, out_features=192, bias=True)
  (activation): LeakyReLU(negative_slope=0.01)
  (message_dropout): Dropout(p=0.3, inplace=False)
  (norm): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
  (classifier): Linear(in_features=192, out_features=2, bias=True)
  (readout): GlobalAttention(gate_nn=Sequential(
    (0): Linear(in_features=192, out_features=96, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=96, out_features=1, bias=True)
  ), nn=None)
  (loss_fn): CrossEntropyLoss()
)


##### **Creating Slide Foundation Model**



In [4]:
from piano.model.slide_encoder import create_slide_encoder

model_name = "chief"

slide_pfm = create_slide_encoder(model_name=model_name)

print(slide_pfm)

CHIEFModel(
  (slide_model): CHIEF(
    (attention_net): Sequential(
      (0): Linear(in_features=768, out_features=512, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.25, inplace=False)
      (3): Attn_Net_Gated(
        (attention_a): Sequential(
          (0): Linear(in_features=512, out_features=256, bias=True)
          (1): Tanh()
          (2): Dropout(p=0.25, inplace=False)
        )
        (attention_b): Sequential(
          (0): Linear(in_features=512, out_features=256, bias=True)
          (1): Sigmoid()
          (2): Dropout(p=0.25, inplace=False)
        )
        (attention_c): Linear(in_features=256, out_features=1, bias=True)
      )
    )
    (classifiers): Linear(in_features=512, out_features=2, bias=True)
    (instance_classifiers): ModuleList(
      (0-1): 2 x Linear(in_features=512, out_features=2, bias=True)
    )
    (instance_loss_fn): CrossEntropyLoss()
    (att_head): Att_Head(
      (fc1): Linear(in_features=512, out_features=256, bias=True)
      (r

#### **Load the image preprocess and text preprocess from the original Patch Foundation Model (Not all models have text preprocess)**

In [10]:

import torch
import torchvision
import torch.nn.functional as F
from PIL import Image
import numpy as np
from piano.model.patch_encoder import create_patch_encoder

# 1. Define the model
# Load the PLIP model from Hugging Face
model = create_patch_encoder("plip")

# Get the model's preprocessing functions if you want
# image_preprocess = model.image_preprocess
text_preprocess = model.text_preprocess

# 2. Load the image
image = Image.open('../img/sample_lusc.png').convert('RGB')  # 256px * 256px resolution
text_labels = ["lung adenocarcinoma", "lung squamous cell carcinoma", "normal"]  # Candidate text labels

# 3. Use the model's preprocessing functions to process the image and text
image_preprocess = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
image_tensor = image_preprocess(image).unsqueeze(0)  # [1, 3, 224, 224]
text_tensors = text_preprocess(text_labels)  # Each label [3, 77]

# 4. Extract features
model.eval()
with torch.inference_mode():
    # Encode image features
    img_feat = model.encode_image(image_tensor)  # [1, C]
    
    # Encode all text features
    text_feats = model.encode_text(text_tensors) # [N, C], N is the number of labels

    # Calculate similarity logits and probabilities
    logits = 100.0 * (img_feat @ text_feats.T)
    probs = logits.softmax(dim=-1)

print("Label probs:", probs)


Label probs: tensor([[0.0868, 0.4174, 0.4958]])


•	**model.encode_image:** Extracts features from the image.

•	**model.encode_text:** Extracts features from the text.

  - For **visual-only models**, `model.encode_image` works *without normalization*, but `text_preprocess` and `model.encode_text` will be *inactive*.
  - For **visual-language models**, both `model.encode_image` and `model.encode_text` are processed *with normalization (F.normalize)*.

**Additional Notes:**

- `model.backbone`: This is the raw model used for feature extraction. You can access it directly if needed.
- `model.get_img_token`: Returns the output, which contains output['patch_tokens'] and output['class_token'].
- `piano.get_model_hf_path(model_name)`: Returns the Hugging Face model path for a given model name. This is useful if you want to load a model checkpoint from Hugging Face.